<a href="https://colab.research.google.com/github/SahilPurabiya/PythonForAstronomy/blob/main/Exoplanet_Parameter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Kepler-452** :    https://science.nasa.gov/exoplanet-catalog/kepler-452-b/

Data retrieved from:    https://mast.stsci.edu/portal/Mashup/Clients/Mast/Portal.html

Fits file: https://drive.google.com/drive/folders/17hNuu7jdh17XD-E2O80jOf_n5T3LIIVf?usp=drive_link

In [ ]:
#import data
from astropy.io import fits
hdul = fits.open("/content/drive/MyDrive/kepler 452 fits files/kplr008311864-2009259160929_llc.fits")

hdul.info()

In [ ]:
#extract light curve data
data = hdul[1].data

In [ ]:
#extract time and flux
time = data['TIME']
flux = data['PDCSAP_FLUX']

In [ ]:
#plot time vs flux
import matplotlib.pyplot as plt

plt.figure()
plt.plot(time, flux, '.', markersize=1)
plt.ylim(64590, 64700)
plt.grid('True')
plt.xlabel("Time (days)")
plt.ylabel("Flux")
plt.show()

In [ ]:
#save time and flux to csv file
import pandas as pd

df = pd.DataFrame({
    'TIME': time,
    'PDCSAP_FLUX': flux
})

df.to_csv('lightcure_kp452b.csv', index="False")

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
import glob

# Get all llc files in folder
files = glob.glob("/content/drive/MyDrive/Kepler/*.fits") #gets all files that match pattern

#to store combined data of time and flux from all quarters
all_time = []
all_flux = []

for file in files:
    hdul = fits.open(file)
    data = hdul[1].data

    time = data['TIME']
    flux = data['PDCSAP_FLUX']

    # Remove NaNs
    mask = ~np.isnan(flux)
    time = time[mask]
    flux = flux[mask]

    # Normalize EACH quarter separately
    flux = flux / np.median(flux)

    all_time.append(time)
    all_flux.append(flux)

    hdul.close()

# Combine all quarters
all_time = np.concatenate(all_time)
all_flux = np.concatenate(all_flux)

# Create dataframe
df = pd.DataFrame({
    "TIME": all_time,
    "NORMALIZED_FLUX": all_flux
})

# Sort by time (important)
df = df.sort_values("TIME")

# Save as CSV
df.to_csv("kepler_452b_all_quarters.csv", index=False)

print("Done. File saved.")


In [ ]:
# === 1. Define period ===
period = 384.84  # Kepler-452b orbital period (days)

# === 2. Compute phase ===
phase = (all_time % period) / period

# === 3. Sort by phase ===
sorted_indices = np.argsort(phase)
phase_sorted = phase[sorted_indices]
flux_sorted = all_flux[sorted_indices]

# === 4. Create bins, to reduce noise from the data ===
bins = 200
phase_bins = np.linspace(0, 1, bins)
digitized = np.digitize(phase_sorted, phase_bins)

binned_phase = []
binned_flux = []

for i in range(1, len(phase_bins)):
    mask = digitized == i
    if np.any(mask):
        binned_phase.append(np.mean(phase_sorted[mask]))
        binned_flux.append(np.mean(flux_sorted[mask]))

binned_phase = np.array(binned_phase)
binned_flux = np.array(binned_flux)

# === 5. Plot ===
plt.figure(figsize=(8,4))
plt.plot(phase_sorted, flux_sorted, '.', markersize=1, alpha=0.2)
plt.plot(binned_phase, binned_flux, 'r-', linewidth=2)
plt.ylim(0.998, 1.002)
plt.xlabel("Phase")
plt.ylabel("Normalized Flux")
plt.title("Phase-folded Light Curve (Binned)")
plt.show()

# === 6. Calculate Transit Depth ===
delta = 1 - np.min(binned_flux)
print('delmin = ', np.min(binned_flux))
print('delmax = ', np.max(binned_flux))
print("Transit depth =", delta)
print("Transit depth (ppm) =", delta * 1e6)


In [ ]:
F_out_1 = np.median(binned_flux)
delta_1 = F_out_1 - np.min(binned_flux)
print('F_out_1 = ', F_out_1)
print("Transit depth =", delta_1)